# 00 · What IKET is, and your first profile

Kernel-level timing tells you a GEMM took 24 µs. It does not tell you which *phase* of that GEMM consumed the time, whether your TMA warp and MMA warp actually overlapped, or which wait is the one holding everything up.

IKET (In-Kernel Event Tracing) answers those questions by emitting named, timestamped events **from inside the kernel**, per warp. This notebook installs it, instruments nothing yourself, and gets you a trace you can open.

## 1. What are needed


| Piece | Role |
|---|---|
| **CuTe DSL** | Write GPU kernels in Python with hardware abstractions |
| **IKET** | Annotate kernel phases with ranges and marks |
| **`run-iket`** | Collect a per-warp trace from an annotated kernel |
| **Perfetto** | Explore the trace interactively |

`iket` and `run-iket` ship *inside* the `nvidia-cutlass-dsl` wheel, so `pip install nvidia-cutlass-dsl` gets you the DSL, `iket` instrumentation support and the profiler (`run-iket`).

### Setup

Every notebook starts here. If this machine cannot profile, the collection cells below print their command instead of failing, and the notebook still reads end to end.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "tools"))
import check_env, nbhelp, trace_summary as ts

env = check_env.report()

Environment
----------------------------------------------------------
  GPU           NVIDIA B300 SXM6 AC  [SM103 (Blackwell)]
  Driver        595.58.03
  cutlass-dsl   4.7.1
  iket API      yes
  run-iket      yes
----------------------------------------------------------
  Live profiling: AVAILABLE


## 2. The instrumentation

The API is five calls. That is the entire surface.

| Call | Meaning |
|---|---|
| `iket.range_start(name)` → token | Open a range, get an explicit handle |
| `iket.range_end(token)` | Close that range |
| `iket.range_push(name)` | Open a range, nesting-style |
| `iket.range_pop()` | Close the innermost range |
| `iket.mark(name, payload)` | A single timestamped point |


Use `push`/`pop` when ranges nest naturally. Use `start`/`end` when the pairing crosses control flow and an explicit handle is clearer.

**Ranges cost more than marks.** An async *issue* — a TMA copy, an MMA — is a point in time, not a duration; making it a mark rather than a range costs one event instead of two and keeps the trace buffer far from overflow. Reserve ranges for real waits and real phases.

In [2]:
nbhelp.figure("nb00/iket_api_surface.png", "The IKET call surface.")

Here is that API in a real kernel. This is the TMA (producer) warp of the warp-specialized GEMM — `range_push` around the phase and the wait, `mark` for the async issue:

In [3]:
nbhelp.figure("nb00/gemm2_tma_warp_instrumented.png", "Producer warp: ranges for waits and phases, a mark for the TMA issue.")

And this is the MMA (consumer) warp of the warp-specialized GEMM.

In [4]:
nbhelp.figure("nb00/gemm2_mma_warp_instrumented.png", "Consumer warp: cons_wait is a range because it is a real wait; mma_issue is a mark.")

Rather than trust a screenshot, read the instrumentation out of the file you are about to profile:

In [5]:
nbhelp.show_iket_calls(
    nbhelp.kernel("instrumented", "fp16_gemm_2_iket_detailed_marks.py"),
    limit=14,
)

fp16_gemm_2_iket_detailed_marks.py: 22 IKET call sites (showing first 14)
------------------------------------------------------------------------
>  121      cute.experimental.iket.range_push("prologue")                       # prologue
       ...
>  309      cute.experimental.iket.range_pop()                                  # prologue
       ...
>  320              cute.experimental.iket.range_push("tma_iter")               # tma_iter
       ...
>  322              cute.experimental.iket.range_push("prod_acquire")           # prod_acquire
       ...
>  324              cute.experimental.iket.range_pop()                          # prod_acquire
       ...
>  327              cute.experimental.iket.mark("tma_issue")  # async op -> mark (no artificial range)
       ...
>  342              cute.experimental.iket.range_pop()                          # tma_iter
       ...
>  361                  cute.experimental.iket.range_push("mma_iter")           # mma_iter
       ...
>  363           

## 3. Run it

One command. `run-iket` wraps your normal Python invocation — it runs the app once to discover kernels, then again with instrumentation active.

In [2]:
OUT = "out/nb00_gemm2"
cmd = (
    f"run-iket --output-dir {OUT} --clobber profile --postprocess all -- "
    f"python {nbhelp.kernel("instrumented", "fp16_gemm_2_iket_detailed_marks.py")} --mnk 4096,4096,512"
)
check_env.run_or_show(cmd, env, est="about 1 minute")

$ run-iket --output-dir out/nb00_gemm2 --clobber profile --postprocess all -- python /path/to/cutedsl-iket-tutorial-notebooks --mnk 4096,4096,512

[run-iket] ============= Dry run application first to collect some info... =============
Running Blackwell fp16 GEMM example 2 with:
  mnk:       [4096, 4096, 512]
  tolerance: 0.1

PASS
[run-iket] Auto-computed context buffer size: 0.00 GB (1,187,020 bytes)
[run-iket] ============= Run application for real profiling... =============
Running Blackwell fp16 GEMM example 2 with:
  mnk:       [4096, 4096, 512]
  tolerance: 0.1

PASS
[run-iket] Dumped perfetto trace to /path/to/cutedsl-iket-tutorial-notebooks
[run-iket] Dumped json trace to /path/to/cutedsl-iket-tutorial-notebooks
[run-iket] Dumped compressed trace to /path/to/cutedsl-iket-tutorial-notebooks
[run-iket] Dumped perfetto HTML viewer to /path/to/cutedsl-iket-tutorial-notebooks
[run-iket] To view: cd /path/to/cutedsl-iket-tutorial-notebooks && python3 -m http.server 8080 , then open 

0

## 4. The trace

`--postprocess all` writes four files. The one to open first is the `.pftrace`: drag it onto [ui.perfetto.dev](https://ui.perfetto.dev/).

| File | Use |
|---|---|
| `iket_pid_*.pftrace` | Perfetto UI — start here |
| `iket_pid_*.trace.json` | Scripted analysis |
| `iket_pid_*.pftrace.gz` | Compressed, for sharing |
| `iket_pid_*.html` | Self-contained viewer |


In [ ]:
nbhelp.figure("nb00/profile_report_overview.png", "What run-iket --postprocess all produces.")

Each warp gets its own track. A **Warp Life Time** bar spans the warp's existence; your named ranges nest underneath it. Because every warp is traced separately, roles that run concurrently appear as parallel bands rather than being averaged together.

### Reading it as numbers

For anything quantitative, use the JSON. `trace_summary` aggregates per-range durations across every warp:

In [3]:
try:
    trace = ts.load(ts.find_trace("out/nb00_gemm2"))
    ts.print_summary(ts.summarize(trace), top=10)
except FileNotFoundError:
    print("No local trace here — the reference numbers below were measured on a NVIDIA B300.")

kernel_cutlass_kernel_TiledMMA_ThrLayoutVMNK21111000_PermutationMNK____MMAAtom_ThrID21_ShapeMNK25625616_TVLayoutA2128161281256_TVLayoutB2128161281256_TVLayoutC21282561281256_cutlasscutenv_0   grid 32x16x1  block 192x1x1  warps traced 3072
warp-lifetime span: 23.4 us

range                      count      median         p90       total
--------------------------------------------------------------------
acc_wait                    2048      2.784u      2.944u    5691.68u
epi_iter                   16384      0.128u      0.160u    2155.26u
prologue                    3072      0.288u      0.320u     909.22u
tma_iter                    4096      0.096u      0.384u     657.18u
prod_acquire                4096      0.064u      0.352u     499.42u
r2s                        16384      0.032u      0.032u     441.70u
mma_iter                    2048      0.160u      0.352u     403.87u
cons_wait                   2048      0.096u      0.288u     261.63u
t2r                        16384      0.00

## 5. The finding

Measured on a NVIDIA B300 at `mnk=4096,4096,512` — 3,072 warps traced, 23.8 µs span:

| Range | Median | Count | What it is |
|---|---|---|---|
| `acc_wait` | **2.752 µs** | 2,048 | Epilogue warp blocked on the MMA commit |
| `prologue` | 0.288 µs | 3,072 | smem/barrier/descriptor setup, uniform across warps |
| `epi_iter` | 0.128 µs | 16,384 | One epilogue subtile |
| `tma_iter` | 0.096 µs | 4,096 | One producer K-tile iteration |


`acc_wait` dominates every other range by an order of magnitude — and it is **not a problem**. It is the epilogue warp idling in the shadow of the mainloop, exactly as warp specialization intends. That distinction, between a wait that costs you and a wait that is by design, is what the rest of this series is about.

**Timestamps are nanoseconds, not cycles.** Values landing on multiples of 32 are the giveaway.

**Next:** [01 · Pipeline basics](01_pipeline_basics.ipynb)

---
*Measured on a single NVIDIA B300, public `nvidia-cutlass-dsl` 4.7.1. Your absolute numbers will differ; the shapes in the trace are the transferable part.*